# 1장. AI와 함께하는 데이터 분석의 시작

LLM 기반 데이터 분석 실무 입문 과정의 실습 노트북입니다.


## 학습 목표

- 주요 개념과 분석 흐름을 이해합니다.
- 제공된 실습 데이터를 불러와 기본 구조를 확인합니다.
- LLM을 분석 보조 도구로 활용하는 방법을 익힙니다.


## 실습 배경

이번 장의 내용을 실습하면서 데이터 로드, 탐색, 전처리, 시각화의 기본 흐름을 확인합니다.


In [11]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data/raw')
sns.set_theme(style='whitegrid')

customers = pd.read_csv(DATA_DIR / 'customers.csv')
customers.head()

,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-06-19
1,2,김정호,F,32,대구,2025-11-02
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
4,5,이예원,F,19,부산,2024-09-13


## 데이터 불러오기 기본 설정
(위에서 함께 진행함)

In [ ]:
customers = pd.read_csv(DATA_DIR / 'customers.csv')
customers.head()

## 실습 진행

```text
customers.csv, orders.csv, order_items.csv를 이용해 
최근 1년 completed 주문 기준 지역별 주문당 평균 상품 수량을 계산하는 pandas 코드를 작성해줘. 
 
컬럼: 
- customers: customer_id, city 
- orders: order_id, customer_id, order_date, order_status 
- order_items: order_id, quantity 
 
주의: 
- order_items는 한 주문에 여러 행이 있을 수 있으므로 order_id별 quantity를 먼저 합산 
- customer_id, order_id로 파일 연결, city별 completed 주문 수와 평균 상품 수량 계산

### 실습 과제

1. 이번 장에서 배운 내용을 바탕으로 분석 질문 3개를 작성
    1. 최근 1년 동안 completed 주문을 기준으로 지역별 고객의 1인당 평균 주문 횟수에는 어떤 차이가 있는가?
    2. 최근 1년 동안 completed 주문을 기준으로 지역별 고객의 1인당 평균 구매 금액에는 어떤 차이가 있는가?
    3. 최근 1년 동안 completed 주문을 기준으로 지역별 한 주문에서 구매하는 평균 상품 수량에는 어떤 차이가 있는가?
2. 3번 질문을 코드로 구현
```

In [7]:
import pandas as pd

c = pd.read_csv("../data/raw/customers.csv")
o = pd.read_csv("../data/raw/orders.csv", parse_dates=["order_date"])
i = pd.read_csv("../data/raw/order_items.csv").groupby("order_id", as_index=False)["quantity"].sum()

result = (o[(o.order_status == "completed") & (o.order_date >= o.order_date.max() - pd.DateOffset(years=1))]
          .merge(c, on="customer_id")
          .merge(i, on="order_id")
          .groupby("city")
          .agg(completed_orders=("order_id", "nunique"), avg_quantity=("quantity", "mean"))
          .sort_values("avg_quantity", ascending=False))

print(result)

      completed_orders  avg_quantity
city                                
대구                  10      9.600000
고양                  18      9.333333
대전                  14      9.000000
부산                  13      8.538462
수원                  14      8.000000
광주                  17      7.941176
울산                  20      7.700000
성남                  31      7.290323
인천                  18      6.722222
서울                  29      6.655172


```text
3. 결과 해석:
    1. 결과 관찰: 최근 1년 completed 주문 기준 지역별 주문당 평균 상품 수량을 비교한 결과 대구가 평균 9.6개로 가장 높았고 고양 9.33개로 나타났으며, 서울이 약 0.66개로 가장 낮았다.
    2. 나의 해석과 판단: 지역에 따라 한 번 주문 시 구매하는 평균 상품 수량에 차이가 있는 것으로 보였다. 대구, 고양 등이 주문 1건 당 구매 수량이 상대적으로 높았고, 서울과 인천이 비교적 낮게 나타났다. 다만 평균값만으로는 특정 지역의 고객이 대량 구매를 선호한다고 단정하기 어렵다. 특히 가장 높은 대구의 경우 주문 건수가 10개로 작기 때문에 일부 높은 값에 의한 혼동이 나타났을 가능성도 있다. 따라서 지역별 평균 상품 수량뿐만 아니라 주문 건수 + 상품 수량 분포 역시 같이 판단해야겠다는 필요성을 느꼈다.
    3. 업무 분석적 판단: 이 결과를 통해 지역별로 한 번 주문 시 구매하는 상품 수량에 차이가 있는지 확인할 수 있으며, 주문 당 구매 수량이 높은 지역 패턴이 지속적으로 확인되면 대량 구매 할인 등 마케팅 전략을 검토할 때 자료로 사용할 수 있다.
    4. 한계와 추가 확인 사항: 다만 지역별 주문 건수가 다르기 때문에 평균값이 정확한 지표라고 보기 어렵다. 일부 주문 영향으로 평균값이 크게 달라질 수 있다. 또한 지역과 구매 수량 사이에 높은 상관관계가 있는지, 통계적으로 유의한 값인지 역시 검토가 필요하다.

```